## 1. Train GIST-LIKE + XGBOOST

In [ ]:
# GIST Features + XGBoost Training

import os
import pandas as pd
import numpy as np
import joblib

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

# PATH CONFIG
gist_csv = r"D:\alzheimer detection.v1i.folder\Fitur_Ekstraksi_Klasik\features_extracted\features_gist_like_multiblock.csv"
model_dir = r"D:\alzheimer detection.v1i.folder\Dashboard\src\classical\model"
os.makedirs(model_dir, exist_ok=True)
save_path = os.path.join(model_dir, "gist_xgb_model.pkl")

# LOAD DATA
if not os.path.exists(gist_csv):
    raise FileNotFoundError(f"[FATAL] File GIST tidak ditemukan: {gist_csv}")

df = pd.read_csv(gist_csv)

# Ambil hanya kolom numerik sebagai fitur
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if not numeric_cols:
    raise RuntimeError("[FATAL] Tidak ada kolom numerik pada fitur GIST")

X = df[numeric_cols].fillna(0)
y = df["label"]

# ENCODING & SCALING
le = LabelEncoder()
y_encoded = le.fit_transform(y)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# SPLIT DATA (60% train, 20% val, 20% test)
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X_scaled, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=0.25,
    random_state=42,
    stratify=y_trainval
)

# XGBOOST TRAINING
xgb_model = XGBClassifier(
    n_estimators=500,
    max_depth=7,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss",
    random_state=42,
    use_label_encoder=False
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=False
)

# EVALUATION HISTORY
evals_result = xgb_model.evals_result()

train_loss = evals_result["validation_0"]["mlogloss"]
val_loss   = evals_result["validation_1"]["mlogloss"]

# Approximasi akurasi dari loss (opsional)
train_acc = [1 - l / max(train_loss) for l in train_loss]
val_acc   = [1 - l / max(val_loss) for l in val_loss]

# SAVE MODEL (.pkl)
joblib.dump({
    "model": xgb_model,
    "scaler": scaler,
    "label_encoder": le,
    "X_val": X_val,
    "y_val": y_val,
    "X_test": X_test,
    "y_test": y_test,
    "train_loss": train_loss,
    "val_loss": val_loss,
    "train_acc": train_acc,
    "val_acc": val_acc
}, save_path)

# SUMMARY
print("\n[SUCCESS] Model GIST XGBoost berhasil disimpan")
print(f"Path model       : {save_path}")
print(f"Total data       : {len(df)}")
print(f"Dimensi fitur GIST: {X.shape[1]}")

c:\Python312\Lib\site-packages\xgboost\training.py:199: UserWarning: [23:38:06] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



[SUCCESS] Model GIST XGBoost berhasil disimpan
Path model : D:\alzheimer detection.v1i.folder\Dashboard\src\classical\model\gist_xgb_model.pkl
Total data : 9766
Dimensi fitur GIST : 768


## 2. Train HOG + XGBOOST

In [ ]:
# HOG Features + XGBoost Training

import os
import pandas as pd
import numpy as np
import joblib

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

# PATH CONFIG
hog_csv = r"D:\alzheimer detection.v1i.folder\Fitur_Ekstraksi_Klasik\features_extracted\features_hog.csv"
model_dir = r"D:\alzheimer detection.v1i.folder\Dashboard\src\classical\model"
os.makedirs(model_dir, exist_ok=True)
save_path = os.path.join(model_dir, "hog_xgb_model.pkl")

# LOAD DATA
if not os.path.exists(hog_csv):
    raise FileNotFoundError(f"[FATAL] File HOG tidak ditemukan: {hog_csv}")

df = pd.read_csv(hog_csv)

# Ambil hanya kolom numerik sebagai fitur
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if not numeric_cols:
    raise RuntimeError("[FATAL] Tidak ada kolom numerik pada fitur HOG")

X = df[numeric_cols].fillna(0)
y = df["label"]

# ENCODING & SCALING
le = LabelEncoder()
y_encoded = le.fit_transform(y)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# SPLIT DATA (60% train, 20% val, 20% test)
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X_scaled, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=0.25,
    random_state=42,
    stratify=y_trainval
)

# XGBOOST TRAINING
xgb_model = XGBClassifier(
    n_estimators=500,
    max_depth=7,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss",
    random_state=42,
    use_label_encoder=False
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=False
)

# TRAINING HISTORY
evals_result = xgb_model.evals_result()

train_loss = evals_result["validation_0"]["mlogloss"]
val_loss   = evals_result["validation_1"]["mlogloss"]

# Approximate accuracy from loss (optional)
train_acc = [1 - l / max(train_loss) for l in train_loss]
val_acc   = [1 - l / max(val_loss) for l in val_loss]

# SAVE MODEL (.pkl)
joblib.dump({
    "model": xgb_model,
    "scaler": scaler,
    "label_encoder": le,
    "X_val": X_val,
    "y_val": y_val,
    "X_test": X_test,
    "y_test": y_test,
    "train_loss": train_loss,
    "val_loss": val_loss,
    "train_acc": train_acc,
    "val_acc": val_acc
}, save_path)

# SUMMARY
print("\n[SUCCESS] Model HOG XGBoost berhasil disimpan")
print(f"Path model       : {save_path}")
print(f"Total data       : {len(df)}")
print(f"Dimensi fitur HOG: {X.shape[1]}")

c:\Python312\Lib\site-packages\xgboost\training.py:199: UserWarning: [23:41:02] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



[SUCCESS] Model HOG XGBoost berhasil disimpan
Path model : D:\alzheimer detection.v1i.folder\Dashboard\src\classical\model\hog_xgb_model.pkl
Total data : 9766
Dimensi fitur HOG : 8100


## 3. Train Hu moment + XGBOOST

In [ ]:
# HU Moments Features + XGBoost Training

import os
import pandas as pd
import numpy as np
import joblib

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

# PATH CONFIG
hu_csv = r"D:\alzheimer detection.v1i.folder\Fitur_Ekstraksi_Klasik\features_extracted\features_hu_moments_multi_patch.csv"
model_dir = r"D:\alzheimer detection.v1i.folder\Dashboard\src\classical\model"
os.makedirs(model_dir, exist_ok=True)
save_path = os.path.join(model_dir, "hu_xgb_model.pkl")

# LOAD DATA
if not os.path.exists(hu_csv):
    raise FileNotFoundError(f"[FATAL] File HU Moments tidak ditemukan: {hu_csv}")

df = pd.read_csv(hu_csv)

# Ambil hanya kolom numerik sebagai fitur
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if not numeric_cols:
    raise RuntimeError("[FATAL] Tidak ada kolom numerik pada fitur HU Moments")

X = df[numeric_cols].fillna(0)
y = df["label"]

# ENCODING & SCALING
le = LabelEncoder()
y_encoded = le.fit_transform(y)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# SPLIT DATA (60% train, 20% val, 20% test)
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X_scaled, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=0.25,
    random_state=42,
    stratify=y_trainval
)

# XGBOOST TRAINING
xgb_model = XGBClassifier(
    n_estimators=500,
    max_depth=7,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss",
    random_state=42,
    use_label_encoder=False
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=False
)

# TRAINING HISTORY
evals_result = xgb_model.evals_result()

train_loss = evals_result["validation_0"]["mlogloss"]
val_loss   = evals_result["validation_1"]["mlogloss"]

# Approximasi akurasi dari loss
train_acc = [1 - l / max(train_loss) for l in train_loss]
val_acc   = [1 - l / max(val_loss) for l in val_loss]

# SAVE MODEL (.pkl)
joblib.dump({
    "model": xgb_model,
    "scaler": scaler,
    "label_encoder": le,
    "X_val": X_val,
    "y_val": y_val,
    "X_test": X_test,
    "y_test": y_test,
    "train_loss": train_loss,
    "val_loss": val_loss,
    "train_acc": train_acc,
    "val_acc": val_acc
}, save_path)

# SUMMARY
print("\n[SUCCESS] Model HU Moments XGBoost berhasil disimpan")
print(f"Path model           : {save_path}")
print(f"Total data           : {len(df)}")
print(f"Dimensi fitur HU Moments : {X.shape[1]}")

c:\Python312\Lib\site-packages\xgboost\training.py:199: UserWarning: [00:28:07] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



[SUCCESS] Model HU Moments XGBoost berhasil disimpan
Path model : D:\alzheimer detection.v1i.folder\Dashboard\src\classical\model\hu_xgb_model.pkl
Total data : 9766
Dimensi fitur HU Moments : 112
